In [2]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import tensorflow as tf
tf.random.set_seed(42)

Download the data sets and normalise the data
For MNIST: 
    - Use min-max scaling, so that the values are in range[0,1]
For CIFAR-10:
    -Use Z score normalisation (mean=0, stdev=1)

In [2]:
from tensorflow.keras.datasets import mnist, cifar10
import numpy as np

(x_raw, y_raw), (x_test_raw, y_test)= mnist.load_data()

val_size = int(x_raw.shape[0] * 0.3)
rng = np.random.default_rng(42)  # seed for reproducibility
idx = np.arange(x_raw.shape[0])
rng.shuffle(idx)

val_idx = idx[:val_size]
train_idx = idx[val_size:]

(x_train_raw, y_train), (x_val_raw, y_val) = (x_raw[train_idx], y_raw[train_idx]), (x_raw[val_idx], y_raw[val_idx])


(x_raw_cif, y_cif), (x_test_cif_raw, y_test_cif) = cifar10.load_data()

val_size_cif = int(x_raw_cif.shape[0]*0.3)
idx_cif = np.arange(x_raw_cif.shape[0])
rng.shuffle(idx_cif)

val_idx_cif = idx_cif[:val_size_cif]
train_idx_cif = idx_cif[val_size_cif:]

(x_train_cif_raw, y_train_cif), (x_val_cif_raw, y_val_cif) = (x_raw_cif[train_idx_cif], y_cif[train_idx_cif]), (x_raw_cif[val_idx_cif], y_cif[val_idx_cif])

x_train_cif_type = x_train_cif_raw.astype("float32")
x_test_cif_type = x_test_cif_raw.astype("float32")
x_validation_cif_type = x_val_cif_raw.astype("float32")

In [3]:
import matplotlib.pyplot as plt
import numpy as np
    
x_train, x_test, x_val= x_train_raw/255.0, x_test_raw/255.0, x_val_raw/255.0

def z_score_normalisation(train, test, validation):
    mean = np.mean(train, axis= (0,1,2), keepdims=True)
    std = np.std(train, axis= (0,1,2), keepdims=True)

    return (train - mean) / std, (test - mean) / std, (validation -mean) /std


x_train_cif, x_test_cif, x_val_cif = z_score_normalisation(x_train_cif_type, x_test_cif_type, x_validation_cif_type)



MNIST. 
Train (with Adam optimizer) a CNN with:
    -First layer: Convolution layer wtih 16-64 filters (3x3), ReLu activation and batch normalisation
    -Second layer: Max pooling (2x2)
    -Third layer: Convolution layer with 32-128 filters (3x3), ReLu activation and batch normalisation
    -Fourth layer: max pooling (2x2)
    -Fifth layer: (after flattening the output from fourth later): A fully connected ("dense") layer with 128 neurons, ReLu activation (and dropout during training with suitable dropout range (somewhere in [0.2, 0.5]))
    -Sixth layer: a layer with 10 neurons (the number of classes) and a softmax activation function

In [45]:
print(x_train.dtype, x_train.shape)
print("min/max:", x_train.min(), x_train.max())


float64 (60000, 28, 28)
min/max: 0.0 1.0


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
model = models.Sequential([
    layers.Input(shape = (28,28,1)),
    #1
    layers.Conv2D(filters=16, kernel_size = (3,3)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    #2
    layers.MaxPooling2D((2,2)),

    #3
    layers.Conv2D(filters=32, kernel_size=(3,3)),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #4
    layers.MaxPooling2D((2,2)),

    #5
    layers.Flatten(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #6
    layers.Dense(10, activation = "softmax")
])

model.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted = model.fit(x_train, y_train, epochs = 10, batch_size = 64, validation_data = (x_val, y_val))

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy: ", test_acc)


Epoch 1/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.9166 - loss: 0.2705 - val_accuracy: 0.9777 - val_loss: 0.0742
Epoch 2/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9706 - loss: 0.0981 - val_accuracy: 0.9813 - val_loss: 0.0622
Epoch 3/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9775 - loss: 0.0749 - val_accuracy: 0.9857 - val_loss: 0.0485
Epoch 4/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9812 - loss: 0.0610 - val_accuracy: 0.9842 - val_loss: 0.0576
Epoch 5/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9834 - loss: 0.0533 - val_accuracy: 0.9844 - val_loss: 0.0544
Epoch 6/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9849 - loss: 0.0460 - val_accuracy: 0.9813 - val_loss: 0.0704
Epoch 7/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9869 - loss: 0.0402 - val_accuracy: 0.9871 - val_loss: 0.0504
Epoch 8/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9886 - loss: 0.0369 - val_ac

In [49]:
print(x_train_cif.shape)
print(y_train_cif.shape)
print(y_train_cif[:5])

(50000, 32, 32, 3)
(50000, 1)
[[6]
 [9]
 [9]
 [4]
 [1]]


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
model_cif = models.Sequential([
    layers.Input(shape = (32,32,3)),
    #1
    layers.Conv2D(filters=16, kernel_size = (3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #2
    layers.MaxPooling2D((2,2)),

    #3
    layers.Conv2D(filters=32, kernel_size=(3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #4
    layers.MaxPooling2D((2,2)),

    #5
    layers.Conv2D(filters=32, kernel_size=(3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #6
    layers.MaxPooling2D((2,2)),

    #5
    layers.Flatten(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #6
    layers.Dense(10, activation = "softmax")
])

model_cif.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted_cif = model_cif.fit(x_train_cif, y_train_cif, epochs = 10, batch_size = 64, validation_data = (x_validation_cif, y_validation_cif))

test_loss_cif, test_acc_cif = model_cif.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 34s 36ms/step - accuracy: 0.4260 - loss: 1.5749 - val_accuracy: 0.5343 - val_loss: 1.2870
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 38ms/step - accuracy: 0.5579 - loss: 1.2289 - val_accuracy: 0.6036 - val_loss: 1.0933
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 32s 41ms/step - accuracy: 0.6098 - loss: 1.0939 - val_accuracy: 0.6621 - val_loss: 0.9535
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 32s 41ms/step - accuracy: 0.6468 - loss: 1.0030 - val_accuracy: 0.6386 - val_loss: 1.0262
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 31s 40ms/step - accuracy: 0.6661 - loss: 0.9451 - val_accuracy: 0.6958 - val_loss: 0.8562
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 32s 40ms/step - accuracy: 0.6855 - loss: 0.8967 - val_accuracy: 0.6897 - val_loss: 0.8955
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 39ms/step - accuracy: 0.6973 - loss: 0.8634 - val_accuracy: 0.7010 - val_loss: 0.8548
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 32s 41ms/step - accuracy: 0.7084 - loss: 0.8283 - 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
model_cif = models.Sequential([
    layers.Input(shape = (32,32,3)),
    #1
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #2
    layers.MaxPooling2D((2,2)),

    #3
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #4
    layers.MaxPooling2D((2,2)),

    #5
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #6
    layers.MaxPooling2D((2,2)),

    #5
    layers.Flatten(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #6
    layers.Dense(10, activation = "softmax")
])

model_cif.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted_cif = model_cif.fit(x_train_cif, y_train_cif, epochs = 10, batch_size = 64, validation_data = (x_validation_cif, y_validation_cif))

test_loss_cif, test_acc_cif = model_cif.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 43s 51ms/step - accuracy: 0.4572 - loss: 1.4921 - val_accuracy: 0.6084 - val_loss: 1.1159
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 45s 58ms/step - accuracy: 0.6011 - loss: 1.1270 - val_accuracy: 0.6286 - val_loss: 1.0544
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 44s 56ms/step - accuracy: 0.6541 - loss: 0.9825 - val_accuracy: 0.6963 - val_loss: 0.8613
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 61ms/step - accuracy: 0.6866 - loss: 0.8985 - val_accuracy: 0.7114 - val_loss: 0.8083
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7093 - loss: 0.8317 - val_accuracy: 0.7181 - val_loss: 0.8125
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 46s 59ms/step - accuracy: 0.7245 - loss: 0.7828 - val_accuracy: 0.7381 - val_loss: 0.7529
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 65ms/step - accuracy: 0.7456 - loss: 0.7273 - val_accuracy: 0.7263 - val_loss: 0.8120
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 65ms/step - accuracy: 0.7607 - loss: 0.6940 - 

In [52]:
import tensorflow as tf
from tensorflow.keras import layers, models
model_cif2 = models.Sequential([

    layers.Input(shape = (32,32,3)),
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1),
    
    #1
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #2
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #3
    layers.MaxPooling2D((2,2)),

    #4
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #5
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #6
    layers.MaxPooling2D((2,2)),

    #7
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    #8
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    #9
    layers.MaxPooling2D((2,2)),

    #10
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #11
    layers.Dense(10, activation = "softmax")
])

model_cif2.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted_cif2 = model_cif2.fit(x_train_cif, y_train_cif, epochs = 10, batch_size = 64, validation_data = (x_validation_cif, y_validation_cif))

test_loss_cif2, test_acc_cif2 = model_cif2.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif2)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 101s 122ms/step - accuracy: 0.3898 - loss: 1.6482 - val_accuracy: 0.4043 - val_loss: 1.6681
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 116s 149ms/step - accuracy: 0.5324 - loss: 1.2998 - val_accuracy: 0.4211 - val_loss: 1.6070
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 93s 119ms/step - accuracy: 0.5937 - loss: 1.1529 - val_accuracy: 0.5401 - val_loss: 1.2894
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 100s 128ms/step - accuracy: 0.6271 - loss: 1.0607 - val_accuracy: 0.5724 - val_loss: 1.2228
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 106s 136ms/step - accuracy: 0.6540 - loss: 0.9824 - val_accuracy: 0.5749 - val_loss: 1.1865
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 106s 135ms/step - accuracy: 0.6772 - loss: 0.9302 - val_accuracy: 0.6511 - val_loss: 1.0154
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 88s 113ms/step - accuracy: 0.6937 - loss: 0.8832 - val_accuracy: 0.6370 - val_loss: 1.0812
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 98s 125ms/step - accuracy: 0.7145 - lo

In the one before this we added: 
    -More filters (num feature maps it represents), if they are too few it cant represent all the distinct features needed to differentiate the different objects
    -More convolution layers before pooling helps get a better representation of what features the image has before we compress it
    -GlobalAveragePooling2D instead of flatten: I averages the feature map for each colour instead of creating a big vector with the values for all heights, widths and colours. This means fewer parameters and less sensitivity tot he exact location of a feature
    -Added dropout in more locations: Randomly removes some data. Avoids overfitting by forcing it to not rely too much on any single activation
    -Data augmentation bcs it creates "extra data" by teaching the model to handle objects that are shifted slightly or mirrored

In [ ]:
import tensorflow as tf
tf.random.set_seed(42)

In [ ]:

from tensorflow.keras import callbacks
model_cif = models.Sequential([

    layers.Input(shape = (32,32,3)),
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1),
    
    #1
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #2
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #3
    layers.MaxPooling2D((2,2)),

    #4
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #5
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #6
    layers.MaxPooling2D((2,2)),

    #7
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    #8
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    #9
    layers.MaxPooling2D((2,2)),

    #10
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #11
    layers.Dense(10, activation = "softmax")
])

model_cif.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
early_stopping = callbacks.EarlyStopping(monitor = "val_loss", patience = 4, restore_best_weights = True)
fitted_cif = model_cif.fit(x_train_cif, y_train_cif, epochs = 10, batch_size = 64, validation_data = (x_test_cif, y_test_cif), callbacks = [early_stopping])

test_loss_cif, test_acc_cif = model_cif.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif)

Implement kNN models for both.
Use the standard euclidean distance measure (applied pixel-by picel)
Run each model with different values of k (from 1-25 eg.). Pick the k with the highers validation accuracy.
Use that to obtain testing accuracy